In [ ]:
# %pip install pymodbus
# 위 주석을 풀고 먼저 pymodbus 라이브러리를 설치해주세요.

from pymodbus.client import ModbusTcpClient
import time as tt
import threading
import random

# 클라이언트 연결
client = ModbusTcpClient('210.119.14.58', port=502)
client.connect()

True

In [ ]:
from pymodbus.client import ModbusTcpClient
import time
import random

class EmergencyStopException(Exception):
    """비상 정지 시 발생시킬 커스텀 예외"""
    pass

class StackerCrane:
    
    # ---------------------------------------------------------
    # 1. 주소 매핑 (최신 이미지 반영)
    # ---------------------------------------------------------
    # Discrete Inputs
    DI_AT_LOAD   = 0
    DI_AT_LEFT   = 1
    DI_AT_RIGHT  = 2
    DI_AT_MIDDLE = 3
    DI_MOVING_X  = 4
    DI_MOVING_Z  = 5
    DI_ESTOP     = 6  
    DI_START     = 7  
    DI_RESET     = 8  

    # Coils
    COIL_LOAD_CONV   = 0
    COIL_FORKS_LEFT  = 1
    COIL_FORKS_RIGHT = 2
    COIL_LIFT        = 3
    COIL_UNLOAD_CONV = 4
    COIL_EXIT_CONV   = 5  
    COIL_ENTRY_CONV  = 7
    COIL_EMITTER     = 8
    COIL_REMOVER     = 9

    # Holding Registers
    HR_TARGET_POS = 0

    def __init__(self, ip, port=502):
        self.client = ModbusTcpClient(ip, port=port)
        if self.client.connect():
            print(f"[{ip}] Modbus 서버 연결 성공")
        else:
            print(f"[{ip}] Modbus 서버 연결 실패")

    def close(self):
        self.client.close()
        print("Modbus 연결 해제")

    # ---------------------------------------------------------
    # 2. 통신 및 안전 감시(Safety) 도우미
    # ---------------------------------------------------------
    def _read_di(self, addr):
        return self.client.read_discrete_inputs(addr).bits[0]

    def _write_coil(self, addr, value):
        self.client.write_coil(addr, value)

    def _write_hr(self, addr, value):
        self.client.write_register(addr, value)

    def check_estop(self):
        if self._read_di(self.DI_ESTOP) == False:
            self._emergency_halt()
            raise EmergencyStopException("\n🚨 [경고] 비상 정지(E-Stop) 버튼이 눌렸습니다! 시스템을 강제 종료합니다.")

    def _emergency_halt(self):
        """비상 정지 시 모든 컨베이어와 구동부를 즉시 정지시킵니다."""
        self._write_coil(self.COIL_LOAD_CONV, False)
        self._write_coil(self.COIL_UNLOAD_CONV, False)
        self._write_coil(self.COIL_EXIT_CONV, False) 
        self._write_coil(self.COIL_FORKS_LEFT, False)
        self._write_coil(self.COIL_FORKS_RIGHT, False)
        self._write_coil(self.COIL_ENTRY_CONV, False)
        self._write_coil(self.COIL_EMITTER, False)
        self._write_coil(self.COIL_REMOVER, False)

    def wait_with_safety(self, condition_func):
        while condition_func():
            self.check_estop()
            time.sleep(0.1)

    # ---------------------------------------------------------
    # 3. 조작반(Control Panel) 버튼 대기 함수
    # ---------------------------------------------------------
    def wait_for_start(self):
        print("\n▶️ 시작(Start) 버튼을 누르면 공정을 시작합니다...")
        while not self._read_di(self.DI_START):
            self.check_estop() 
            time.sleep(0.1)
        print("▶️ 시작 버튼 감지! 설비를 가동합니다.")

    def wait_for_reset(self):
        print("\n🔄 에러 상태입니다. 해제하려면 리셋(Reset) 버튼을 누르세요...")
        while not self._read_di(self.DI_RESET):
            time.sleep(0.1)
        print("🔄 리셋 완료! 시스템 대기 상태로 복귀합니다.")

    # ---------------------------------------------------------
    # 4. 단위 공정 (입고/출고/이동)
    # ---------------------------------------------------------
    def init_environment(self):
        """[개선] 공장 환경 및 크레인 물리 상태 초기화"""
        print("-> 초기 환경 설정 및 크레인 물리 리셋 중...")
        # 1. 컨베이어 및 에미터 상태 정렬
        self._write_coil(self.COIL_ENTRY_CONV, True)
        self._write_coil(self.COIL_REMOVER, True)
        self._write_coil(self.COIL_EMITTER, False)
        self._write_coil(self.COIL_LOAD_CONV, False)
        self._write_coil(self.COIL_UNLOAD_CONV, False)
        self._write_coil(self.COIL_EXIT_CONV, False)

        # 2. [중요] 리프트 하강 및 포크 센터 복귀 강제 초기화
        self._write_coil(self.COIL_LIFT, False)         # 리프트 내림 (물건을 처음 집을 수 있도록 준비)
        self._write_coil(self.COIL_FORKS_LEFT, False)   # 포크 원위치
        self._write_coil(self.COIL_FORKS_RIGHT, False)  # 포크 원위치
        time.sleep(1) # 기계가 완전히 정지하고 리셋될 물리적 대기 시간

    def move_and_wait(self, target_pos):
        print(f"   -> 타겟 이동 (위치: {target_pos})")
        self._write_hr(self.HR_TARGET_POS, target_pos)

        self.wait_with_safety(lambda: not (self._read_di(self.DI_MOVING_X) or self._read_di(self.DI_MOVING_Z)))
        self.wait_with_safety(lambda: self._read_di(self.DI_MOVING_X) or self._read_di(self.DI_MOVING_Z))
            
        print("   -> 도착 및 안정화(1초)")
        time.sleep(1) 
        self.check_estop()

    def pickup_from_load_conveyor(self):
        print("   -> 포크 뻗기 (왼쪽 입고 컨베이어)")
        self._write_coil(self.COIL_FORKS_LEFT, True)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_LEFT))

        print("   -> 리프트 상승")
        self._write_coil(self.COIL_LIFT, True)
        time.sleep(1) 

        print("   -> 포크 복귀")
        self._write_coil(self.COIL_FORKS_LEFT, False)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_MIDDLE))

    def drop_to_rack(self):
        print("   -> 포크 뻗기 (오른쪽 랙)")
        self._write_coil(self.COIL_FORKS_RIGHT, True)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_RIGHT))

        print("   -> 리프트 하강")
        self._write_coil(self.COIL_LIFT, False)
        time.sleep(1) 

        print("   -> 포크 복귀")
        self._write_coil(self.COIL_FORKS_RIGHT, False)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_MIDDLE))

    def pickup_from_rack(self):
        print("   -> 포크 뻗기 (오른쪽 랙)")
        self._write_coil(self.COIL_FORKS_RIGHT, True)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_RIGHT))

        print("   -> 리프트 상승")
        self._write_coil(self.COIL_LIFT, True)
        time.sleep(1) 

        print("   -> 포크 복귀")
        self._write_coil(self.COIL_FORKS_RIGHT, False)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_MIDDLE))

    def drop_to_unload_conveyor(self):
        print("   -> 포크 뻗기 (오른쪽 출고 컨베이어)")
        self._write_coil(self.COIL_FORKS_RIGHT, True)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_RIGHT))

        print("   -> 리프트 하강")
        self._write_coil(self.COIL_LIFT, False)
        time.sleep(1) 

        print("   -> 포크 복귀")
        self._write_coil(self.COIL_FORKS_RIGHT, False)
        self.wait_with_safety(lambda: not self._read_di(self.DI_AT_MIDDLE))
        
        print("   -> 출고(Unload) 및 배출(Exit) 컨베이어 가동")
        self._write_coil(self.COIL_UNLOAD_CONV, True)
        self._write_coil(self.COIL_EXIT_CONV, True)
        time.sleep(3) 
        self._write_coil(self.COIL_UNLOAD_CONV, False)
        self._write_coil(self.COIL_EXIT_CONV, False)

    # ---------------------------------------------------------
    # 5. 시나리오 실행 메서드
    # ---------------------------------------------------------
    def process_loading(self, target_list):
        print(f"\n[입고 작업 시작] 타겟: {target_list}")
        self._write_coil(self.COIL_ENTRY_CONV, True)
        self._write_coil(self.COIL_REMOVER, True)

        for pos in target_list:
            print(f"\n>>> [입고] 위치 {pos} 작업 시작")
            self._write_coil(self.COIL_EMITTER, True)
            time.sleep(0.5)
            self._write_coil(self.COIL_EMITTER, False)

            self._write_coil(self.COIL_LOAD_CONV, True)

            # 1. 새로운 화물이 다가와서 센서에 감지될 때까지 대기
            self.wait_with_safety(lambda: not self._read_di(self.DI_AT_LOAD))

            self._write_coil(self.COIL_LOAD_CONV, False)

            self.pickup_from_load_conveyor()
            self._write_coil(self.COIL_LOAD_CONV, True) 

            self.move_and_wait(pos)
            self.drop_to_rack()
            self.move_and_wait(99)
            print(f"<<< [입고] 위치 {pos} 완료")

    def process_unloading(self, target_list):
        print(f"\n[출고 작업 시작] 타겟: {target_list}")
        self._write_coil(self.COIL_REMOVER, True)

        for pos in target_list:
            print(f"\n>>> [출고] 위치 {pos} 작업 시작")
            self.move_and_wait(pos)
            self.pickup_from_rack()
            self.move_and_wait(99)
            self.drop_to_unload_conveyor()
            print(f"<<< [출고] 위치 {pos} 완료")


# =====================================================================
# 메인 실행부
# =====================================================================
if __name__ == "__main__":
    crane = StackerCrane(ip='210.119.14.58')

    try:
        # [해결책 1] 루프 진입 전 반드시 설비 및 크레인 초기화를 먼저 수행합니다.
        crane.init_environment()

        while True:
            try:
                # 1. Start 버튼 대기
                crane.wait_for_start()

                # 2. 1번부터 54번까지 순차 입고 생성
                # load_targets = [1, 5, 12] 
                load_targets = list(range(1, 55))
                random.shuffle(load_targets)

                crane.process_loading(load_targets)

                # 3. 54번부터 1번까지 역순 출고 생성
                # unload_targets = [5, 1] 
                unload_targets = list(range(54, 0, -1))
                random.shuffle(unload_targets)

                crane.process_unloading(unload_targets)

                print("\n✅ 모든 할당 작업이 완료되었습니다. 다음 시작(Start)을 대기합니다.")

            except EmergencyStopException as es:
                print(es)
                crane.wait_for_reset()
                # [안전 예방] 비상정지 후 리셋 버튼이 눌리면, 기계를 다시 완전히 초기화합니다.
                crane.init_environment()

    except KeyboardInterrupt:
        print("\n[알림] 사용자에 의해 프로그램이 강제 종료되었습니다.")
    except Exception as e:
        print(f"시스템 오류 발생: {e}")
    finally:
        crane.close()

[210.119.14.58] Modbus 서버 연결 성공
-> 초기 환경 설정 및 크레인 물리 리셋 중...

▶️ 시작(Start) 버튼을 누르면 공정을 시작합니다...
▶️ 시작 버튼 감지! 설비를 가동합니다.

[입고 작업 시작] 타겟: [22, 16, 34, 23, 24, 33, 7, 17, 51, 35, 43, 26, 2, 29, 32, 14, 50, 37, 47, 11, 46, 13, 45, 3, 53, 1, 5, 25, 15, 40, 31, 12, 30, 42, 9, 4, 39, 18, 8, 20, 41, 52, 19, 36, 38, 6, 44, 10, 28, 27, 49, 48, 54, 21]

>>> [입고] 위치 22 작업 시작
   -> 포크 뻗기 (왼쪽 입고 컨베이어)
   -> 리프트 상승
   -> 포크 복귀
   -> 타겟 이동 (위치: 22)
   -> 도착 및 안정화(1초)
   -> 포크 뻗기 (오른쪽 랙)
   -> 리프트 하강
   -> 포크 복귀
   -> 타겟 이동 (위치: 99)
   -> 도착 및 안정화(1초)
<<< [입고] 위치 22 완료

>>> [입고] 위치 16 작업 시작
   -> 포크 뻗기 (왼쪽 입고 컨베이어)
   -> 리프트 상승
   -> 포크 복귀
   -> 타겟 이동 (위치: 16)
   -> 도착 및 안정화(1초)
   -> 포크 뻗기 (오른쪽 랙)
   -> 리프트 하강
   -> 포크 복귀
   -> 타겟 이동 (위치: 99)
   -> 도착 및 안정화(1초)
<<< [입고] 위치 16 완료

>>> [입고] 위치 34 작업 시작
   -> 포크 뻗기 (왼쪽 입고 컨베이어)
   -> 리프트 상승
   -> 포크 복귀
   -> 타겟 이동 (위치: 34)
   -> 도착 및 안정화(1초)
   -> 포크 뻗기 (오른쪽 랙)
   -> 리프트 하강
   -> 포크 복귀
   -> 타겟 이동 (위치: 99)
   -> 도착 및 안정화(1초)
<<< [입고] 위치 34 완료

>>> [입고] 위